[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_01/MFM_ch1_vol_estimators/MFM_ch1_vol_estimators.ipynb)

# MFM_ch1_vol_estimators

Range-based volatility estimators on S&P 500 OHLC data (2008-2026): close-to-close, Parkinson, Garman-Klass, Rogers-Satchell and Yang-Zhang, around the COVID-19 crash, and the relative smoothness of their daily updates; stale opens of the S&P 500 index compared with the SPY ETF.

*Modelling Financial Markets (MFM), Chapter 1: Stylised Facts and Returns. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_1'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
import warnings
warnings.filterwarnings('ignore')

## Style and helpers

In [ ]:
# Chart style: transparent background, legend below the plot
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Chart colours
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'
LightGray = '#DADADA'
Teal     = '#17A2B8'

ASSETS = ['sp500', 'bettr', 'btc', 'eurron', 'gold']
COLORS = {'sp500': MainBlue, 'bettr': IDAred, 'btc': Amber, 'eurron': Forest, 'gold': Purple}
PERIODS = {'sp500': 252, 'bettr': 252, 'btc': 365, 'eurron': 252, 'gold': 252}


def save_fig(name):
    """Save the figure as PDF and transparent PNG in the current folder."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Place the legend outside the plot, bottom centre."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')

SOURCES = {
    'sp500':  ('GSPC.INDX',    'close', '1990-01-01'),
    'bettr':  ('BETTR.INDX',   'close', '2014-09-01'),
    'btc':    ('BTC-USD.CC',   'close', '2014-09-17'),
    'eurron': ('REF:EUR',      'close', '2005-07-01'),   # from the introduction of the new leu (RON), 1 July 2005
    'gold':   ('XAUUSD.FOREX', 'close', '1990-01-01'),
}

LABELS = {'sp500': 'S&P 500', 'bettr': 'BET-TR (Romania)', 'btc': 'Bitcoin',
          'eurron': 'EUR/RON', 'gold': 'Gold'}

# OTC quotes (gold): partial Saturday/Sunday sessions break the Friday -> Monday return,
# so we keep weekdays only (the same convention as in Chapter 0)
WEEKDAYS_ONLY = {'gold'}

_CACHE = {}


def read_market(symbol):
    """Read the daily price table of one symbol (local copy or GitHub)."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    df = pd.read_csv(src, parse_dates=['date']).set_index('date').sort_index()
    df.index.name = 'Date'
    return df


def read_reference_rate(currency='EUR', start='2005-07-01', end='2026-09-18'):
    """Official RON reference rate (BNR annual XML archives)."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    import re
    import urllib.request
    rows = []
    for y in range(int(start[:4]), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}">([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    df = pd.DataFrame(rows, columns=['Date', 'Close']).drop_duplicates('Date').set_index('Date')
    df.index = pd.to_datetime(df.index)
    _CACHE[key] = df.sort_index().loc[start:end]
    return _CACHE[key]


def load_data(name='sp500'):
    """Daily OHLCV with columns Open/High/Low/Close/Volume (Close = the price used)."""
    symbol, col, start = SOURCES[name]
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start)
    d = read_market(symbol).loc[start:]
    if name in WEEKDAYS_ONLY:
        d = d[d.index.dayofweek < 5]
    out = pd.DataFrame({'Open': d['open'], 'High': d['high'], 'Low': d['low'],
                        'Close': d[col], 'Volume': d['volume']})
    return out[out['Close'] > 0].dropna(subset=['Close'])


def load_close(name):
    """Closing price, used in all charts."""
    return load_data(name)['Close'].dropna()


closes = {a: load_close(a) for a in ASSETS}
rets = {a: np.log(c).diff().dropna() for a, c in closes.items()}
# gold (weekdays only) is annualised with its actual number of observations per year
PERIODS['gold'] = len(rets['gold']) / ((closes['gold'].index[-1] - closes['gold'].index[0]).days / 365.25)
spx_ohlc = load_data('sp500')
for a in ASSETS:
    print(f'{LABELS[a]:18s}', rets[a].index[0].date(), '->', rets[a].index[-1].date(), len(rets[a]), 'obs')

## Helper functions

In [ ]:
def vol_close_to_close(df, n=21, periods=252):
    r = np.log(df['Close']).diff()
    return r.rolling(n).std() * np.sqrt(periods)


def vol_parkinson(df, n=21, periods=252):
    hl = np.log(df['High'] / df['Low']) ** 2
    return np.sqrt(hl.rolling(n).mean() / (4 * np.log(2)) * periods)


def vol_garman_klass(df, n=21, periods=252):
    hl = np.log(df['High'] / df['Low']) ** 2
    co = np.log(df['Close'] / df['Open']) ** 2
    v = 0.5 * hl - (2 * np.log(2) - 1) * co
    return np.sqrt(v.rolling(n).mean() * periods)


def vol_rogers_satchell(df, n=21, periods=252):
    ho, hc = np.log(df['High'] / df['Open']), np.log(df['High'] / df['Close'])
    lo, lc = np.log(df['Low'] / df['Open']), np.log(df['Low'] / df['Close'])
    return np.sqrt((ho * hc + lo * lc).rolling(n).mean() * periods)


def vol_yang_zhang(df, n=21, periods=252):
    o = np.log(df['Open'] / df['Close'].shift(1))          # overnight
    c = np.log(df['Close'] / df['Open'])                   # open-to-close
    ho, hc = np.log(df['High'] / df['Open']), np.log(df['High'] / df['Close'])
    lo, lc = np.log(df['Low'] / df['Open']), np.log(df['Low'] / df['Close'])
    rs = (ho * hc + lo * lc).rolling(n).mean()
    k = 0.34 / (1.34 + (n + 1) / (n - 1))
    v = o.rolling(n).var() + k * c.rolling(n).var() + (1 - k) * rs
    return np.sqrt(v * periods)

## Analysis

In [ ]:
def fig_vol_estimators():
    # before 2007 the index open equals the previous close (data artefact) -> use 2008-2026
    d = spx_ohlc.loc['2008-01-01':]
    est = pd.DataFrame({
        'Close-to-close': vol_close_to_close(d), 'Parkinson': vol_parkinson(d),
        'Garman-Klass': vol_garman_klass(d), 'Rogers-Satchell': vol_rogers_satchell(d),
        'Yang-Zhang': vol_yang_zhang(d)}).dropna()
    seg = est.loc['2019-07-01':'2021-06-30']
    fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.0), gridspec_kw={'width_ratios': [1.6, 1]})
    cols = [Teal, MainBlue, Forest, Amber, IDAred]
    for c, col in zip(est.columns, cols):
        axes[0].plot(seg.index, 100 * seg[c], color=col, lw=0.9, label=c)
    import matplotlib.dates as mdates
    axes[0].xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
    axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    axes[0].tick_params(axis='x', labelsize=7, rotation=30)
    axes[0].set_ylabel('Annualised volatility (%, 21-day)')
    axes[0].set_title('S&P 500 around the COVID-19 crash', fontsize=8.5, loc='left')
    axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.28), ncol=3, frameon=False, fontsize=7)
    # relative efficiency: weekly variability of the estimator (noise) vs close-to-close
    noise = est.diff().std()
    eff = (noise['Close-to-close'] / noise) ** 2
    axes[1].barh(eff.index, eff.values, color=cols, alpha=0.85)
    axes[1].axvline(1, color=Gray, ls=':', lw=0.8)
    axes[1].set_xlabel('Relative smoothness vs close-to-close')
    axes[1].set_title('Noise of daily updates', fontsize=8.5, loc='left')
    axes[1].tick_params(axis='y', labelsize=7.5)
    plt.tight_layout()
    save_fig('ch1_vol_estimators')
    means = 100 * est.mean()
    return pd.DataFrame({'mean_pct': means, 'rel_smoothness': eff})


def stale_open_check(start='2008-01-01'):
    """Compare the S&P 500 index with the SPY ETF (open = traded price)."""
    g = spx_ohlc.loc[start:]
    spy = read_market('SPY.US').loc[start:]
    f = spy['adjusted_close'] / spy['close']
    s = pd.DataFrame({'Open': spy['open'] * f, 'High': spy['high'] * f, 'Low': spy['low'] * f,
                      'Close': spy['adjusted_close']})
    rows = []
    for name, d in [('S&P 500 index', g), ('SPY ETF', s)]:
        on = np.log(d['Open'] / d['Close'].shift(1))
        oc = np.log(d['Close'] / d['Open'])
        rows.append({'series': name, 'close_to_close': vol_close_to_close(d).mean(),
                     'yang_zhang': vol_yang_zhang(d).mean(), 'parkinson': vol_parkinson(d).mean(),
                     'overnight_vol': on.std() * np.sqrt(252), 'open_to_close_vol': oc.std() * np.sqrt(252),
                     'cov_share': 2 * on.cov(oc) / np.log(d['Close']).diff().var(),
                     'open_eq_prev_close': (np.abs(d['Open'] / d['Close'].shift(1) - 1) < 1e-6).mean()})
    t = pd.DataFrame(rows).set_index('series')
    o = spx_ohlc['Open']; c = spx_ohlc['Close'].shift(1)
    eq = (np.abs(o / c - 1) < 1e-6).groupby(spx_ohlc.index.year).mean()
    t.to_csv(os.path.join(TABLE_DIR, 'ch1_stale_open.csv'), float_format='%.4g')
    eq.to_csv(os.path.join(TABLE_DIR, 'ch1_stale_open_by_year.csv'), float_format='%.4g')
    return t, eq

## Run

In [ ]:
TABLE_DIR = '.'
print(fig_vol_estimators().round(3))
so, so_year = stale_open_check()
print(so.round(3))

![ch1_vol_estimators](./ch1_vol_estimators.png)

Output: `ch1_vol_estimators.pdf`